## 1. Install SageMaker SDK


In [ ]:
%pip install -q --upgrade -r requirements.txt

%pip uninstall -y sagemaker sagemaker-core sagemaker-train sagemaker-serve sagemaker-mlops

%pip install --no-cache-dir --force-reinstall "sagemaker>=2.245.0,<3"

In [ ]:
#RESTART THE KERNEL AFTER YOU RUN THE CELL ABOVE

## 2. Imports and SageMaker setup


In [ ]:
import base64
import io
import json
import shutil
import sys
import tarfile
import time
from urllib.parse import urlparse

import boto3
import matplotlib.pyplot as plt
import numpy as np
import requests
import sagemaker
from sagemaker import image_uris
from sagemaker.huggingface import HuggingFace, HuggingFaceModel
from sagemaker.model_monitor import DataCaptureConfig
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer

sys.path.append('scripts')

session = sagemaker.Session()
boto_session = boto3.Session()
REGION = boto_session.region_name
ROLE = sagemaker.get_execution_role()
BUCKET = session.default_bucket()

print({'region': REGION, 'bucket': BUCKET, 'role': ROLE})


## 3. Deploy SageMaker endpoint with data capture


In [ ]:
PROJECT = 'recycling-classifier-course'
ENDPOINT_NAME = f'{PROJECT}-endpoint'
DATA_CAPTURE_S3_URI = f's3://{BUCKET}/{PROJECT}/monitoring/data-capture/v1'
DEPLOY_INSTANCE_TYPE = 'ml.m5.large'
TRANSFORMERS_VERSION = '4.37.0'
PYTORCH_VERSION = '2.1.0'
PY_VERSION = 'py310'

MODEL_DATA_S3_URI = ''

model = HuggingFaceModel(
    model_data=MODEL_DATA_S3_URI,
    role=ROLE,
    transformers_version=TRANSFORMERS_VERSION,
    pytorch_version=PYTORCH_VERSION,
    py_version=PY_VERSION,
    entry_point='inference.py',
    source_dir='scripts',
)

data_capture_config = DataCaptureConfig(
    enable_capture=True,
    sampling_percentage=100,
    destination_s3_uri=DATA_CAPTURE_S3_URI,
    capture_options=['REQUEST', 'RESPONSE'],
)

predictor = model.deploy(
    initial_instance_count=1,
    instance_type=DEPLOY_INSTANCE_TYPE,
    endpoint_name=ENDPOINT_NAME,
    data_capture_config=data_capture_config,
)
predictor.serializer = JSONSerializer()
predictor.deserializer = JSONDeserializer()
print('endpoint', ENDPOINT_NAME)


## 4. Test SageMaker predictor


In [ ]:
import requests
import base64
import json

url = ""

img_bytes = requests.get(url).content
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

result = predictor.predict({"image": img_b64})

if isinstance(result, list):
    result = result[0]

if isinstance(result, bytes):
    result = result.decode("utf-8")

result = json.loads(result)

print("\nPrediction")
print("=" * 40)
print(f"Predicted label : {result['predicted_label']}")
print(f"Label ID        : {result['predicted_label_id']}")
print(f"Confidence      : {result['confidence']:.2%}")
print(f"Image size      : {result['width']} x {result['height']}")

print("\nTop predictions")
print("=" * 40)

for i, pred in enumerate(result["top_predictions"], start=1):
    print(
        f"{i}. {pred['label']:<15} "
        f"{pred['probability']:.2%}"
    )


## 5. Label names


In [ ]:
label_names = [
    'aluminium',
    'batteries',
    'cardboard',
    'disposable plates',
    'glass',
    'hard plastic',
    'paper',
    'paper towel',
    'polystyrene',
    'soft plastics',
    'takeaway cups'
]


## 6. Lambda environment variables


In [ ]:
lambda_client = boto3.client('lambda', region_name=REGION)

PROJECT = 'recycling-classifier-course'
ENDPOINT_NAME = f'{PROJECT}-endpoint'
PIPELINE_NAME = f'{PROJECT}-pipeline'
LOW_CONFIDENCE_PREFIX = f'{PROJECT}/low-confidence'
DEPLOY_INSTANCE_TYPE = 'ml.m5.large'
TRANSFORMERS_VERSION = '4.37.0'
PYTORCH_VERSION = '2.1.0'
PY_VERSION = 'py310'
BEDROCK_MODEL_ID = 'global.anthropic.claude-sonnet-4-6'
CONFIDENCE_THRESHOLD = '0.65'
MIN_RETRAIN_ITEMS = '3'

LAMBDA_FUNCTION_NAME = ''

api_lambda_environment = {
    'SAGEMAKER_ENDPOINT_NAME': ENDPOINT_NAME,
    'LOW_CONFIDENCE_BUCKET': BUCKET,
    'LOW_CONFIDENCE_PREFIX': LOW_CONFIDENCE_PREFIX,
    'PIPELINE_NAME': PIPELINE_NAME,
    'CONFIDENCE_THRESHOLD': str(CONFIDENCE_THRESHOLD),
    'MIN_RETRAIN_ITEMS': str(MIN_RETRAIN_ITEMS),
    'BEDROCK_MODEL_ID': BEDROCK_MODEL_ID,
    'ALLOWED_LABELS_JSON': json.dumps(label_names),
    'METRIC_NAMESPACE': 'RecyclingClassifierCourse',
}

current_config = lambda_client.get_function_configuration(
    FunctionName=LAMBDA_FUNCTION_NAME
)

current_environment = current_config.get('Environment', {}).get('Variables', {})


## 7. Update Lambda configuration


In [ ]:
updated_environment = {
    **current_environment,
    **api_lambda_environment,
}

lambda_client.update_function_configuration(
    FunctionName=LAMBDA_FUNCTION_NAME,
    Environment={
        'Variables': updated_environment
    }
)


## 8. IAM setup for Lambda role


In [ ]:
import json
import boto3

REGION = 'eu-central-1'
LAMBDA_FUNCTION_NAME = ''

lambda_client = boto3.client('lambda', region_name=REGION)
iam_client = boto3.client('iam')
sts_client = boto3.client('sts')

account_id = sts_client.get_caller_identity()['Account']

lambda_config = lambda_client.get_function_configuration(
    FunctionName=LAMBDA_FUNCTION_NAME
)

lambda_role_name = lambda_config['Role'].split('/')[-1]

endpoint_arn = f'arn:aws:sagemaker:{REGION}:{account_id}:endpoint/{ENDPOINT_NAME}'
pipeline_arn = f'arn:aws:sagemaker:{REGION}:{account_id}:pipeline/{PIPELINE_NAME}'
bucket_arn = f'arn:aws:s3:::{BUCKET}'
low_confidence_object_arn = f'arn:aws:s3:::{BUCKET}/{LOW_CONFIDENCE_PREFIX.strip("/")}/*'


## 9. IAM policy document


In [ ]:
policy_document = {
    'Version': '2012-10-17',
    'Statement': [
        {
            'Effect': 'Allow',
            'Action': 'sagemaker:InvokeEndpoint',
            'Resource': endpoint_arn,
        },
        {
            'Effect': 'Allow',
            'Action': 'sagemaker:StartPipelineExecution',
            'Resource': pipeline_arn,
        },
        {
            'Effect': 'Allow',
            'Action': [
                's3:PutObject',
                's3:GetObject',
                's3:DeleteObject',
            ],
            'Resource': low_confidence_object_arn,
        },
        {
            'Effect': 'Allow',
            'Action': 's3:ListBucket',
            'Resource': bucket_arn,
            'Condition': {
                'StringLike': {
                    's3:prefix': f'{LOW_CONFIDENCE_PREFIX.strip("/")}/*'
                }
            },
        },
        {
            'Effect': 'Allow',
            'Action': [
                'bedrock:InvokeModel',
                'bedrock:Converse',
            ],
            'Resource': '*',
        },
    ],
}

iam_client.put_role_policy(
    RoleName=lambda_role_name,
    PolicyName='RecyclingClassifierApiPermissions',
    PolicyDocument=json.dumps(policy_document),
)

print('updated role:', lambda_role_name)




## 10. Test API Gateway endpoint


In [ ]:
API_URL = ""
url = ""

test_threshold = 0.65
img_bytes = requests.get(url, timeout=120).content

api_payload = {
    'image': base64.b64encode(img_bytes).decode('utf-8'),
    'content_type': 'image/jpeg',
    'threshold': test_threshold,
}

response = requests.post(API_URL, json=api_payload, timeout=120)
print(response.status_code)
print(response.text[:4000])

## 11. Updating our Sagemaker Pipeline


In [ ]:
inference_image_uri = image_uris.retrieve(
    framework="huggingface",
    region=REGION,
    version=TRANSFORMERS_VERSION,
    image_scope="inference",
    instance_type=DEPLOY_INSTANCE_TYPE,
    base_framework_version=f"pytorch{PYTORCH_VERSION}",
    py_version=PY_VERSION,
)

DEPLOY_LAMBDA_ARN = (
    ""
)
from pipeline_definition import get_pipeline

pipeline = get_pipeline(
    REGION, ROLE, BUCKET, DEPLOY_LAMBDA_ARN, inference_image_uri, scripts_dir="scripts"
)
pipeline.upsert(role_arn=ROLE)